In [29]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image, Markdown
from alerce.core import Alerce
import os
import numpy as np
alerce = Alerce()

In [30]:
SN_IDS = [
    "ZTF22abhwlnm", "ZTF22aadesjc", "ZTF22abfdzrv", "ZTF21abgkfzh",
    "ZTF23aatcsou", "ZTF23aadjssg", "ZTF18aaiwzie", "ZTF23aaekebt",
    "ZTF23aadbtou", "ZTF22abghrui", "ZTF21abiggqx", "ZTF23aaefpfb",
    "ZTF22aatwxrl", "ZTF23aaahnss", "ZTF22absuavp", "ZTF23aafgmaz",
    "ZTF23aaflptz", "ZTF23aatcola", "ZTF23aajestr", "ZTF23aahjdxa",
    "ZTF22abrbohu", "ZTF23aarzzwu", "ZTF22abzajwl", "ZTF23aagxvad",
    "ZTF23aaempzk"
]

In [31]:
def build_features_dataframe(sn_ids):
    data_list = []
    for oid in sn_ids:
        try:
            feats = alerce.query_features(oid, format="pandas")
            if feats is not None and not feats.empty:
                feat_dict = feats.set_index("name")["value"].to_dict()
                feat_dict["oid"] = oid
                data_list.append(feat_dict)
        except Exception as e:
            print(f"Error fetching features for {oid}: {e}")
    return pd.DataFrame(data_list)

In [32]:
def map_features_to_physical(df):
    mapped = df.copy()
    rename_dict = {
        "mag_max": "mag_max",
        "t_rise": "t_rise",
        "dm15": "dm15",
        "Mean": "Mean",
        "g-r_mean": "g-r_mean",
        "MaxSlope": "MaxSlope"
    }
    mapped.rename(columns={old: new for old, new in rename_dict.items() if old in df.columns}, inplace=True)
    return mapped

In [33]:
def fetch_typeIa_reference(n_samples=20):
    try:
        typeIa_sn = alerce.query_objects(
            classifier="lc_classifier",
            class_name="SNIa",
            probability_threshold=0.7,
            format="pandas"
        )
        typeIa_sn = typeIa_sn.head(n_samples)
        feats_list = []
        for oid in typeIa_sn['oid']:
            feats = alerce.query_features(oid, format="pandas")
            if feats is not None and not feats.empty:
                feat_dict = feats.set_index("name")["value"].to_dict()
                feat_dict["oid"] = oid
                feats_list.append(feat_dict)
        return pd.DataFrame(feats_list) if feats_list else pd.DataFrame()
    except Exception as e:
        print("Error fetching Type Ia SNe:", e)
        return pd.DataFrame()


In [34]:
OUTLIER_THRESHOLD = 0.2
def compute_difference_and_flag(df, reference_props):
    diff_table = df.copy()
    for prop, ref_value in reference_props.items():
        if prop in df.columns:
            diff_table[f"{prop}_diff"] = df[prop] - ref_value
            diff_table[f"{prop}_outlier"] = diff_table[f"{prop}_diff"].abs() > OUTLIER_THRESHOLD
    return diff_table

In [35]:
def extract_outliers(diff_table):
    outlier_cols = [col for col in diff_table.columns if col.endswith("_outlier")]
    diff_cols = [col for col in diff_table.columns if col.endswith("_diff")]
    outlier_table = diff_table.copy()
    mask = outlier_table[outlier_cols].any(axis=1)
    outlier_table = outlier_table.loc[mask, ["oid"] + diff_cols + outlier_cols]
    return outlier_table

In [36]:
def plot_comparison(df, typeIa_avg, properties=["peak_mag","rise_time"]):
    for prop in properties:
        if prop in df.columns:
            plt.figure(figsize=(6,4))
            plt.scatter(df['oid'], df[prop], s=80, color='blue', label='Your SNs')
            plt.axhline(typeIa_avg[prop], color='red', linestyle='--', label='Type Ia Average')
            plt.xticks(rotation=45)
            plt.ylabel(prop)
            plt.title(f'{prop} Comparison to Type Ia Average')
            plt.legend()
            plt.tight_layout()
            plt.show()

In [37]:
def main():
    print(f"Using {len(SN_IDS)} supernovae IDs.")

    df = build_features_dataframe(SN_IDS)
    if df.empty:
        print("No features available for your input SNs.")
        return
    
    df_mapped = map_features_to_physical(df)

    # Keep only classification-relevant features
    columns_to_show = ["oid", "mag_max", "t_rise", "dm15", "Mean", "g-r_mean", "MaxSlope"]
    df_filtered = df_mapped[[col for col in columns_to_show if col in df_mapped.columns]]

    display(Markdown("## Key Supernova Features for Classification"))
    display(df_filtered)

    # Fetch Type Ia reference
    typeIa_df = fetch_typeIa_reference(n_samples=20)
    if typeIa_df.empty:
        print("Cannot compute Type Ia reference differences. Exiting.")
        return
    typeIa_mapped = map_features_to_physical(typeIa_df)

    # Compute Type Ia averages only for numeric columns we care about
    numeric_cols = ["mag_max", "t_rise", "dm15", "Mean", "g-r_mean", "MaxSlope"]
    typeIa_avg_filtered = typeIa_mapped[numeric_cols].select_dtypes(include=np.number).mean()
    display(Markdown("## Typical Type Ia Average Values"))
    display(pd.DataFrame(typeIa_avg_filtered).T)

    # Compute differences and outliers
    diff_table = compute_difference_and_flag(df_filtered, typeIa_avg_filtered)
    display(Markdown("## Differences from Type Ia & Outlier Flags"))
    display(diff_table)

    outliers = extract_outliers(diff_table)
    display(Markdown("## Only Outlier Features"))
    if not outliers.empty:
        display(outliers)
    else:
        display(Markdown("No outliers detected."))

    # Plot comparison (mag_max and t_rise are most discriminating)
    plot_comparison(df_filtered, typeIa_avg_filtered, properties=["mag_max","t_rise"])

# Run workflow
main()

Using 25 supernovae IDs.


## Key Supernova Features for Classification

,oid,Mean,g-r_mean,MaxSlope
0,ZTF22abhwlnm,NaN,0.100957,NaN
1,ZTF22aadesjc,18.463453,0.311640,0.867480
2,ZTF22abfdzrv,19.629977,0.285459,0.067661
3,ZTF21abgkfzh,NaN,0.459420,NaN
4,ZTF23aatcsou,19.820835,0.190543,0.120993
5,ZTF23aadjssg,18.454111,1.046436,0.081238
6,ZTF18aaiwzie,18.430837,-0.392514,88.263345
7,ZTF23aaekebt,18.922177,0.356721,256.313378
8,ZTF23aadbtou,18.848800,0.143260,0.118470
9,ZTF22abghrui,19.354617,0.356615,177.550690


KeyError: "['mag_max', 't_rise', 'dm15'] not in index"